# Hospital Clock - Colab renderer (code lives on GitHub)
Runtime -> **T4 GPU**. Set `REPO_URL` below, then run top to bottom. First run: `SCALE = 0.5` (preview); then `1.0` for final.

In [ ]:
#@title 1) Setup - Drive, clone repo, installs
import os, sys, json, subprocess, time
from google.colab import drive
drive.mount('/content/drive')
REPO_URL = 'https://github.com/YOUR_USER/YOUR_REPO.git'  #@param {type:"string"}
BRANCH   = 'main'                                        #@param {type:"string"}
STORY    = 'hospital_clock'                              #@param {type:"string"}
PROJECT  = '/content/drive/MyDrive/hospital_clock_video' #@param {type:"string"}
for d in ['backgrounds', 'depth', 'out']: os.makedirs(f'{PROJECT}/{d}', exist_ok=True)
import shutil; shutil.rmtree('/content/repo', ignore_errors=True)
!git clone -q --depth 1 -b {BRANCH} {REPO_URL} /content/repo
!pip -q install -U diffusers transformers accelerate safetensors faster-whisper scipy
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE -> Runtime > Change runtime type > T4 GPU')
os.makedirs('/content/video', exist_ok=True); os.chdir('/content/video')
sys.path[:0] = ['/content/repo/src', '/content/repo/stories']

In [ ]:
%%writefile /content/video/shots.py
# shots.py - the shot list. Every shot is anchored to a WORD INDEX in the script, so when Whisper gives real word
# timings the cuts, punches, expression swaps and pulses all land on the right words automatically.
SCRIPT = ("Long ago, a hospital's big clock stopped, and nobody fixed it. Soon, the head doctor noticed something strange. "
          "Patients were being checked more often than ever. With no clock to watch, the nurses stopped waiting for the hour. "
          "They simply walked the ward, bed by bed. Most people would blame the broken clock and replace it. "
          "But the doctor looked closer. The clock hadn't caused the problem. It had exposed one: "
          "the nurses had been watching the clock, not the patients.")
# word indices: 0 Long | 11 Soon | 18 Patients | 26 With | 38 They | 46 Most | 56 But | 61 The | 67 It | 71 the(nurses) | 78 not

STYLE = ("stylized 3D render, toy-like miniature diorama, vintage hospital, glossy painted materials, soft global "
         "illumination, saturated colors, clean simple shapes, cinematic depth of field, vertical composition, "
         "no people, no characters, no text, no letters")
NEG = "people, person, character, face, text, letters, watermark, logo, blurry, photo, realistic skin"

# ---- backgrounds (7 unique images; shots reuse them with different crops/camera moves) ----------------------------
BACKGROUNDS = {
    "ward_clock_wide": "long old hospital ward seen from the entrance, two rows of iron beds with white sheets, a huge round "
                       "brass wall clock centered on the far wall, warm amber dusty light through tall windows, symmetrical",
    "clock_ecu": "extreme close-up of a giant antique brass wall clock face with roman numerals, frozen hands, dust in a "
                 "shaft of golden light, shallow depth of field, dark warm background",
    "ward_day": "bright cheerful hospital ward in daylight, rows of neat iron beds with teal blankets, big windows, "
                "teal and cream colors, wide empty floor in the foreground, eye level",
    "ward_low_sym": "low angle view down the center aisle of a hospital ward, perfectly symmetrical, rows of beds both sides, "
                    "the big brass clock high on the far wall center, soft light beams, floor in the foreground",
    "ward_topdown": "top-down overhead view of a hospital ward floor, parallel rows of iron beds with teal blankets, "
                    "checkered floor, soft light, diorama",
    "boardroom": "elegant hospital administration hallway with marble floor, red carpet, tall arched windows, wood paneling, "
                 "cool blue morning light with gold accents, empty",
    "spotlight_dark": "dark warm hospital corridor, a single spotlight cone from above on an empty floor, deep brown and amber, "
                      "moody, soft bokeh in the background",
}

def c(kind, x, y, h, **kw):
    d = dict(kind=kind, x=x, y=y, h=h); d.update(kw); return d

W_ = lambda i, off=0.0: ("w", i, off)   # time of word i (+offset)

SHOTS = [
 # 1 HOOK 0.0-1.0  medium-wide, low angle, FAST push-in, then push-through transition into the ECU
 dict(id=1, bg="ward_clock_wide", anchor=0, t=0.00, tin="cut",
      cam=dict(z=(1.0, 1.30), x=(0, 0), y=(0.01, -0.02), ease="out", par=0.9, dof=0.55, focus=(0.35, 0.35), hand=0.4),
      chars=[], fx=dict(dust=True, bloom=0.18, vignette=0.32, grade=(0.95, 1.0, 1.06))),
 # 2 ECU of the stopped clock - push + rack focus (dust sharp -> hands sharp), tiny jolt on "stopped"
 dict(id=2, bg="clock_ecu", anchor=2, t=1.00, tin="push",
      cam=dict(z=(1.0, 1.22), x=(0, 0.01), y=(0, 0), ease="io", par=0.8, dof=0.9, focus=(0.9, 0.35), focus_t=(0.0, ("w", 6)),
               punches=[(W_(6), 0.04)], hand=0.3),
      chars=[], fx=dict(dust=True, dust_amt=0.5, bloom=0.30, vignette=0.38, grade=(0.92, 1.0, 1.1))),
 # 3 staff ignoring the clock - slow lateral dolly, bed rail blurred in the foreground
 dict(id=3, bg="ward_clock_wide", anchor=7, t=3.10, tin="cut",
      cam=dict(z=(1.05, 1.10), x=(-0.05, 0.05), y=(0, 0), ease="lin", par=1.2, dof=0.6, focus=(0.4, 0.4), hand=0.3),
      chars=[c("nurse", 0.28, 0.86, 0.34, depth=0.45, path=((0, 0.20, 0.86), (1.5, 0.40, 0.86)), walk=True),
             c("nurse", 0.74, 0.84, 0.30, depth=0.42, flip=True, path=((0, 0.80, 0.84), (1.5, 0.66, 0.84)), walk=True,
               expr=[(0, "neutral")])],
      fx=dict(dust=True, bloom=0.15, vignette=0.32, grade=(0.95, 1.0, 1.05))),
 # 4 doctor medium, slow push-in ("Soon, the head doctor noticed")
 dict(id=4, bg="ward_day", anchor=11, t=4.60, tin="cut",
      cam=dict(z=(1.0, 1.10), x=(0.02, 0), y=(0, 0), ease="io", par=0.8, dof=0.5, focus=(0.55, 0.55), hand=0.3),
      chars=[c("doctor", 0.46, 0.93, 0.62, depth=0.6, expr=[(0, "neutral"), (W_(15), "think")], pose=[(0, "down")])],
      fx=dict(bloom=0.12, vignette=0.28, grade=(1.04, 1.02, 0.98))),
 # 5 ZOOM PUNCH on "strange": shock face + sweat drop + speed lines
 dict(id=5, bg="ward_day", anchor=16, t=6.35, tin="cut",
      cam=dict(z=(1.18, 1.26), x=(0.0, -0.01), y=(0.015, 0.015), ease="lin", par=0.9, dof=0.7, focus=(0.6, 0.6),
               punches=[(W_(17), 0.20)], hand=0.5),
      chars=[c("doctor", 0.5, 1.02, 0.78, depth=0.6, expr=[(0, "think"), (W_(17), "shock")], pose=[(0, "down"), (W_(17), "up")])],
      events=[dict(type="speed", t=W_(17), strength=0.6)],
      fx=dict(bloom=0.12, vignette=0.30, grade=(1.04, 1.02, 0.98))),
 # 6 patient being checked - medium, push-in on the nurse's check
 dict(id=6, bg="ward_day", anchor=18, t=7.88, tin="cut",
      cam=dict(z=(1.15, 1.30), x=(-0.04, -0.02), y=(0.02, 0.0), ease="io", par=1.0, dof=0.7, focus=(0.55, 0.55), hand=0.3),
      chars=[c("patient_bed", 0.62, 0.78, 0.30, depth=0.55, expr=[(0, "neutral"), (W_(21), "worried")]),
             c("nurse", 0.28, 0.86, 0.46, depth=0.62, expr=[(0, "neutral")], pose=[(0, "down"), (W_(21), "point")])],
      fx=dict(bloom=0.12, vignette=0.28, grade=(1.03, 1.02, 0.99))),
 # 7 REPEATED SHOT: whip to another nurse checking the same patient AGAIN ("more often than ever")
 dict(id=7, bg="ward_day", anchor=22, t=9.20, tin="whip_l",
      cam=dict(z=(1.25, 1.12), x=(0.04, 0.0), y=(0.0, 0.01), ease="io", par=1.0, dof=0.75, focus=(0.5, 0.5), hand=0.5),
      chars=[c("patient_bed", 0.40, 0.80, 0.30, depth=0.55, expr=[(0, "worried"), (W_(25), "weak")]),
             c("nurse", 0.74, 0.88, 0.50, depth=0.65, flip=True, expr=[(0, "neutral")], pose=[(0, "point")])],
      fx=dict(bloom=0.12, vignette=0.30, grade=(1.03, 1.02, 0.99))),
 # 8 nurses under the blank clock, SYMMETRICAL low angle ("With no clock to watch")
 dict(id=8, bg="ward_low_sym", anchor=26, t=10.71, tin="cut",
      cam=dict(z=(1.0, 1.12), x=(0, 0), y=(0.02, -0.01), ease="io", par=0.7, dof=0.5, focus=(0.55, 0.55), hand=0.2),
      chars=[c("nurse", 0.30, 0.90, 0.46, depth=0.55, expr=[(0, "look_up"), (W_(32), "neutral")]),
             c("nurse", 0.70, 0.90, 0.46, depth=0.55, flip=True, expr=[(0, "look_up"), (W_(32), "neutral")])],
      fx=dict(dust=True, bloom=0.2, vignette=0.34, grade=(0.97, 1.0, 1.04))),
 # 9 nurses turn and walk away from the clock ("stopped waiting for the hour")
 dict(id=9, bg="ward_low_sym", anchor=34, t=12.60, tin="cut",
      cam=dict(z=(1.12, 1.0), x=(0.02, -0.02), y=(0, 0), ease="io", par=1.1, dof=0.5, focus=(0.5, 0.5), hand=0.3),
      chars=[c("nurse", 0.30, 0.90, 0.46, depth=0.55, path=((0, 0.30, 0.90), (1.5, 0.12, 0.92)), walk=True),
             c("nurse", 0.70, 0.90, 0.46, depth=0.55, flip=True, path=((0, 0.70, 0.90), (1.5, 0.90, 0.92)), walk=True)],
      fx=dict(dust=True, bloom=0.2, vignette=0.34, grade=(0.97, 1.0, 1.04))),
 # 10 WHIP into a high-angle tracking shot over the ward ("They simply walked the ward,")
 dict(id=10, bg="ward_topdown", anchor=38, t=14.31, tin="whip_r",
      cam=dict(z=(1.0, 1.25), x=(-0.06, 0.06), y=(-0.05, 0.04), rot=(-3, 3), ease="lin", par=1.3, dof=0.35, focus=(0.5, 0.5)),
      chars=[c("nurse", 0.30, 0.55, 0.24, depth=0.5, path=((0, 0.30, 0.52), (1.5, 0.62, 0.62)), walk=True)],
      fx=dict(bloom=0.25, vignette=0.30, grade=(1.0, 1.0, 1.02))),
 # 11 macro "bed by bed" - three glow pulses, one per word, slow top-down push
 dict(id=11, bg="ward_topdown", anchor=43, t=15.87, tin="cut",
      cam=dict(z=(1.3, 1.55), x=(0.05, -0.02), y=(0.04, -0.04), rot=(2, -2), ease="io", par=1.0, dof=0.6, focus=(0.6, 0.6)),
      events=[dict(type="pulse", t=W_(43), pos=(0.32, 0.38)), dict(type="pulse", t=W_(44), pos=(0.66, 0.52)),
              dict(type="pulse", t=W_(45), pos=(0.38, 0.70))],
      chars=[], fx=dict(bloom=0.3, vignette=0.30, grade=(1.0, 1.0, 1.03))),
 # 12 RIDICULE: suits point and laugh at the clock - medium-wide, handheld bob
 dict(id=12, bg="boardroom", anchor=46, t=17.17, tin="cut",
      cam=dict(z=(1.05, 1.15), x=(0.02, -0.02), y=(0, 0), ease="io", par=0.9, dof=0.55, focus=(0.55, 0.55), hand=1.2),
      chars=[c("suit", 0.26, 0.92, 0.50, depth=0.6, expr=[(0, "smug"), (W_(49), "laugh")], pose=[(0, "cross"), (W_(49), "point")]),
             c("suit", 0.62, 0.90, 0.46, depth=0.55, flip=True, expr=[(0, "smug"), (W_(50), "laugh")], pose=[(0, "cross")], wiggle=True),
             c("suit", 0.88, 0.88, 0.40, depth=0.5, flip=True, expr=[(0, "neutral"), (W_(51), "laugh")], pose=[(0, "down")])],
      fx=dict(bloom=0.15, vignette=0.28, grade=(1.05, 1.0, 0.96))),
 # 13 "...and replace it": gleaming new clock rolled in, sparkles + flash
 dict(id=13, bg="boardroom", anchor=53, t=19.00, tin="cut",
      cam=dict(z=(1.1, 1.28), x=(-0.02, 0.01), y=(0.01, -0.01), ease="out", par=0.9, dof=0.6, focus=(0.55, 0.55), hand=0.6),
      chars=[c("prop", 0.5, 0.62, 0.40, name="clock", depth=0.55,
               path=((0, 0.5, 0.80), (0.8, 0.5, 0.62)), path_ease="back"),
             c("suit", 0.20, 0.92, 0.44, depth=0.6, expr=[(0, "smug")], pose=[(0, "cross")])],
      events=[dict(type="flash", t=W_(54))],
      fx=dict(sparkle=dict(pos=(0.5, 0.45), spread=300), bloom=0.4, vignette=0.28, grade=(1.05, 1.02, 0.97))),
 # 14 DISSOLVE to the doctor: push-in to the eyes ("But the doctor looked closer.") + rack focus
 dict(id=14, bg="spotlight_dark", anchor=56, t=20.28, tin="dissolve",
      cam=dict(z=(1.0, 1.35), x=(0, 0), y=(0.03, -0.02), ease="io", par=0.6, dof=0.8, focus=(0.3, 0.7)),
      chars=[c("doctor", 0.5, 1.30, 1.55, depth=0.7, bust=True, expr=[(0, "think"), (W_(59), "squint")])],
      fx=dict(bloom=0.25, vignette=0.45, grade=(0.95, 0.98, 1.05))),
 # 15 ECU of the old clock; slow push, then a long PULL-BACK with rack focus reveals the nurses beneath it
 dict(id=15, bg="clock_ecu", anchor=61, t=22.19, tin="cut",
      cam=dict(z=(1.35, 1.0), x=(0, 0), y=(-0.02, 0.03), ease="io", par=1.1, dof=0.85, focus=(0.9, 0.35), focus_t=(("w", 67), ("w", 71)),
               punches=[]),
      chars=[], fx=dict(dust=True, dust_amt=0.5, bloom=0.3, vignette=0.4, grade=(0.92, 1.0, 1.08))),
 # 16 wide symmetrical: nurses lined up staring at the clock, patients blurred behind
 dict(id=16, bg="ward_low_sym", anchor=71, t=25.86, tin="cut",
      cam=dict(z=(1.0, 1.14), x=(0, 0), y=(0.03, -0.01), ease="io", par=0.8, dof=0.7, focus=(0.6, 0.6), hand=0.15),
      chars=[c("nurse", 0.22, 0.92, 0.46, depth=0.6, expr=[(0, "look_up")]),
             c("nurse", 0.50, 0.94, 0.52, depth=0.62, expr=[(0, "look_up")]),
             c("nurse", 0.78, 0.92, 0.46, depth=0.6, flip=True, expr=[(0, "look_up")]),
             c("patient_bed", 0.50, 0.66, 0.18, depth=0.3, expr=[(0, "weak")])],
      fx=dict(dust=True, bloom=0.22, vignette=0.36, grade=(0.97, 1.0, 1.04))),
 # 17 FINAL HOLD: patient sharp in the foreground, nurses blurred behind; rack focus onto the patient on "patients"
 dict(id=17, bg="ward_low_sym", anchor=78, t=28.03, tin="cut",
      cam=dict(z=(1.35, 1.22), x=(0, 0), y=(0.04, 0.02), ease="out", par=0.9, dof=0.95, focus=(0.55, 0.15), focus_t=(("w", 79), ("w", 80)), hand=0.1),
      chars=[c("nurse", 0.34, 0.84, 0.34, depth=0.62, flip=False, expr=[(0, "look_up")]),
             c("nurse", 0.66, 0.84, 0.34, depth=0.62, flip=True, expr=[(0, "look_up")]),
             c("patient_bed", 0.5, 1.02, 0.60, depth=0.12, expr=[(0, "weak"), (W_(80), "weak")], pose=[(0, "down"), (W_(80), "wave")])],
      fx=dict(bloom=0.2, vignette=0.42, grade=(0.96, 1.0, 1.04))),
]
# transitions OUT of each shot are read from the NEXT shot's "tin"

In [ ]:
#@title 3) Settings
SCALE    = 0.5     #@param {type:"number"}   # 0.5 = fast preview (540x960). 1.0 = final 1080x1920
QUALITY  = 'base'  #@param ["base", "turbo"] # base = SDXL (best look, ~30 s/image). turbo = SDXL-Turbo (fast, softer)
SEED     = 1234    #@param {type:"integer"}
TAIL     = 0.85    #@param {type:"number"}   # seconds the last image holds after the voice ends
SFX      = True    #@param {type:"boolean"}  # soft whooshes on whip cuts + a pop on the zoom punch
SFX_DB   = -20     #@param {type:"number"}
CAP_SIZE = 78      #@param {type:"integer"}  # caption font size at full 1080 width
FORCE    = False   #@param {type:"boolean"}  # re-render every shot even if cached

In [ ]:
#@title 4) Upload the voice-over (or put it at MyDrive/hospital_clock_video/voiceover.mp3)
AUDIO_PATH = f'{PROJECT}/voiceover.mp3'
if not os.path.exists(AUDIO_PATH):
    from google.colab import files
    import shutil
    up = files.upload(); shutil.move(list(up)[0], AUDIO_PATH)
dur = float(subprocess.check_output(['ffprobe', '-v', 'error', '-show_entries', 'format=duration', '-of', 'csv=p=0', AUDIO_PATH]))
print('voice-over length:', round(dur, 2), 's')

In [ ]:
#@title 5) Align the script to the voice-over (Whisper word timestamps) and print the cut sheet
import importlib, engine as E
S = importlib.import_module(STORY)
importlib.reload(E); importlib.reload(S)
words = E.tokenize(S.SCRIPT)
try:
    from faster_whisper import WhisperModel
    try: wm = WhisperModel('small', device='cuda', compute_type='float16')
    except Exception: wm = WhisperModel('small', device='cpu', compute_type='int8')
    segs, _ = wm.transcribe(AUDIO_PATH, language='en', word_timestamps=True)
    asr = [(w.word.strip(), w.start, w.end) for s in segs for w in s.words]
    wt = E.align_words(words, asr, dur)
    print('Whisper heard', len(asr), 'words; script has', len(words))
except Exception as e:
    print('Whisper unavailable -> using pause-based estimate:', e)
    wt = E.estimate_words_from_silence(AUDIO_PATH, words, dur)
json.dump(wt, open(f'{PROJECT}/word_times.json', 'w'))
starts = [0.0 if s['anchor'] == 0 else wt[s['anchor']][0] for s in S.SHOTS]
ends = starts[1:] + [dur + TAIL]
print(f"{'shot':>4} {'start':>6} {'end':>6} {'len':>5}  background        transition-in")
for s, a, b in zip(S.SHOTS, starts, ends):
    print(f"{s['id']:>4} {a:6.2f} {b:6.2f} {b-a:5.2f}  {s['bg']:<16}  {s['tin']}")

In [ ]:
#@title 6) Generate the diorama backgrounds (SDXL) - skips images that already exist
import gc, torch
from diffusers import StableDiffusionXLPipeline, AutoPipelineForText2Image
REGEN = []   # e.g. ['boardroom'] to re-roll one background with a new seed offset
todo = [k for k in S.BACKGROUNDS if k in REGEN or not os.path.exists(f'{PROJECT}/backgrounds/{k}.png')]
print('to generate:', todo)
if todo:
    if QUALITY == 'turbo':
        pipe = AutoPipelineForText2Image.from_pretrained('stabilityai/sdxl-turbo', torch_dtype=torch.float16, variant='fp16').to('cuda')
        size, kw = (512, 896), dict(num_inference_steps=4, guidance_scale=0.0)
    else:
        pipe = StableDiffusionXLPipeline.from_pretrained('stabilityai/stable-diffusion-xl-base-1.0', torch_dtype=torch.float16,
                                                         variant='fp16', use_safetensors=True).to('cuda')
        size, kw = (768, 1344), dict(num_inference_steps=28, guidance_scale=6.5, negative_prompt=S.NEG)
    pipe.enable_vae_slicing()
    for k in todo:
        off = list(S.BACKGROUNDS).index(k) + (1000 if k in REGEN else 0)
        g = torch.Generator('cuda').manual_seed(SEED + off)
        img = pipe(prompt=S.BACKGROUNDS[k] + ', ' + S.STYLE, width=size[0], height=size[1], generator=g, **kw).images[0]
        img.save(f'{PROJECT}/backgrounds/{k}.png'); print('saved', k)
    del pipe; gc.collect(); torch.cuda.empty_cache()

In [ ]:
#@title 7) Depth maps (Depth-Anything-V2) + preview of every background
import numpy as np
from PIL import Image
from transformers import pipeline
import matplotlib.pyplot as plt
dp = pipeline('depth-estimation', model='depth-anything/Depth-Anything-V2-Small-hf', device=0 if torch.cuda.is_available() else -1)
for k in S.BACKGROUNDS:
    p = f'{PROJECT}/depth/{k}.npy'
    if os.path.exists(p) and k not in REGEN: continue
    im = Image.open(f'{PROJECT}/backgrounds/{k}.png').convert('RGB')
    d = np.array(dp(im)['depth']).astype(np.float32)          # brighter = nearer
    np.save(p, (d - d.min()) / (d.max() - d.min() + 1e-6))
del dp; gc.collect(); torch.cuda.empty_cache()
fig, ax = plt.subplots(2, len(S.BACKGROUNDS), figsize=(3 * len(S.BACKGROUNDS), 10))
for i, k in enumerate(S.BACKGROUNDS):
    ax[0, i].imshow(Image.open(f'{PROJECT}/backgrounds/{k}.png')); ax[0, i].set_title(k, fontsize=9)
    ax[1, i].imshow(np.load(f'{PROJECT}/depth/{k}.npy'), cmap='magma')
    ax[0, i].axis('off'); ax[1, i].axis('off')
plt.tight_layout(); plt.show()

In [ ]:
#@title 8) Character preview (procedural 2D cast - identical in every shot)
import characters as ch
combo = [('doctor','neutral','point'),('doctor','shock','up'),('doctor','squint','think'),('nurse','look_up','down'),
         ('nurse','laugh','shrug'),('suit','smug','cross'),('suit','laugh','point'),('patient','weak','down')]
fig, ax = plt.subplots(1, len(combo), figsize=(2.2 * len(combo), 5))
for a, (k, e, p) in zip(ax, combo):
    s = ch.draw_character(k, e, p, 0, 1); a.imshow(s[..., [2, 1, 0, 3]]); a.set_title(f'{k}/{e}', fontsize=8); a.axis('off')
plt.show()

In [ ]:
#@title 9) Render the shots (CPU, cached on Drive). Set ONLY=[5, 11] to redo specific shots.
import cv2, numpy as np
ONLY = []
E.W, E.H = int(1080 * SCALE) // 2 * 2, int(1920 * SCALE) // 2 * 2
SHOT_DIR = f'{PROJECT}/shots_{int(SCALE * 100)}'; os.makedirs(SHOT_DIR, exist_ok=True)
bgs = {k: E.BG(cv2.imread(f'{PROJECT}/backgrounds/{k}.png'), np.load(f'{PROJECT}/depth/{k}.npy')) for k in S.BACKGROUNDS}
files = []; t_all = time.time()
for i, s in enumerate(S.SHOTS):
    p = f"{SHOT_DIR}/shot_{s['id']:02d}.mp4"; meta = p + '.json'; files.append(p)
    want = dict(start=round(starts[i], 3), end=round(ends[i], 3))
    same = os.path.exists(p) and os.path.exists(meta) and json.load(open(meta)) == want
    if (ONLY and s['id'] not in ONLY) or (not ONLY and same and not FORCE):
        continue
    tin = s['tin']; tout = S.SHOTS[i + 1]['tin'] if i + 1 < len(S.SHOTS) else 'cut'
    gp = f'{SHOT_DIR}/ghost_{i-1}.png'
    ghost = cv2.imread(gp) if (tin == 'dissolve' and i > 0 and os.path.exists(gp)) else None
    t0 = time.time()
    n = E.render_shot(s, bgs[s['bg']], wt, starts[i], ends[i], p, tin=tin, tout=tout, ghost=ghost,
                      ghost_save=f'{SHOT_DIR}/ghost_{i}.png')
    json.dump(want, open(meta, 'w'))
    print(f"shot {s['id']:>2}: {n:3d} frames in {time.time()-t0:5.1f}s")
print('done in', round(time.time() - t_all), 's')

In [ ]:
#@title 10) Captions + SFX + voice-over -> final MP4
chunks = E.caption_chunks(words, wt)
whips = [starts[i] for i, s in enumerate(S.SHOTS) if s['tin'] in ('whip_l', 'whip_r', 'push')]
pops = [starts[i] + E.tsec(e['t'], starts[i], wt) for i, s in enumerate(S.SHOTS) for e in s.get('events', []) if e['type'] == 'speed']
E.make_sfx('/content/video/mix.wav', AUDIO_PATH, whips if SFX else [], pops if SFX else [], sfx_db=SFX_DB)
OUT = f'{PROJECT}/out/hospital_clock_{int(SCALE * 100)}p.mp4'
n = E.finalize(files, chunks, '/content/video/mix.wav', OUT, cap_size=int(CAP_SIZE * SCALE))
print('frames:', n, '->', OUT)
from IPython.display import Video, display
display(Video(OUT, embed=True, width=300))
from google.colab import files as colab_files
colab_files.download(OUT)

## Customising
* **Another story:** change `SCRIPT`, `BACKGROUNDS` and `SHOTS` in `shots.py` (cell 2c). Each shot is anchored to a **word index**
  (`anchor`), so cuts, expression swaps, punches and pulses follow the real word timing automatically.
* **Camera per shot (`cam`):** `z` zoom (start, end), `x`/`y` pan, `rot` roll, `ease` (`lin|io|out|in|back`), `par` parallax strength,
  `dof` depth-of-field amount, `focus` (start, end) + `focus_t` for rack focus, `hand` handheld amount, `punches` zoom-punch list.
* **Transitions (`tin`):** `cut`, `whip_l`, `whip_r`, `push` (radial push-through), `dissolve` (ghosted double exposure).
* **Events:** `speed` (speed lines), `pulse` (glow ring), `flash`; `fx` has `dust`, `sparkle`, `bloom`, `vignette`, `grade`.
* **Re-roll one background:** put its name in `REGEN` (cells 6-7), re-run, then re-render the shots that use it.
* **Pacing target from the reference:** a new shot every ~1.5-3 s, fast first 7 s, slow held reveal near the middle, a held final image.